In [1]:
import hashlib
import json
from urllib.parse import urljoin

import requests
from cryptography.hazmat.primitives import padding
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes

In [2]:
EAPI_SERVER = "https://interfacepc.music.163.com/"
EAPI_KEY = b"e82ckenh8dichen8"
EAPI_SEP = "-36cd479b6b5-"
EAPI_DIGEST = "nobody{}use{}md5forencrypt"
EAPI_PLAINTEXT = "{}{}{}{}{}".format("{}", EAPI_SEP, "{}", EAPI_SEP, "{}")

In [3]:
def json_compact(obj: dict | list) -> str:
    return json.dumps(obj, separators=(",", ":"))


def eapi_encrypt(path: str, payload: dict) -> str:

    padder = padding.PKCS7(algorithms.AES.block_size).padder()
    encryptor = Cipher(algorithms.AES(EAPI_KEY), modes.ECB()).encryptor()

    text = json_compact(payload)
    digest = hashlib.md5(EAPI_DIGEST.format(path, text).encode("utf-8")).hexdigest()
    dec = EAPI_PLAINTEXT.format(path, text, digest).encode("utf-8")

    enc = padder.update(dec) + padder.finalize()
    enc = encryptor.update(enc) + encryptor.finalize()
    return enc.hex().upper()


def eapi_post(path: str, data: dict) -> dict:
    r = requests.post(
        urljoin(EAPI_SERVER, "/eapi/" + path.removeprefix("/api/")),
        data={"params": eapi_encrypt(path, data)},
    )
    return r.json()

In [4]:
EFFECTS_PATH = "/api/usertool/sound/queryAllSound"
EFFECTS_EXPOSE_TYPES = (2, 3, 6, 7)


def fetch_effects_catalog() -> dict[int, list[dict]]:
    return {
        i: eapi_post(EFFECTS_PATH, {"exposeType": i})["data"]
        for i in EFFECTS_EXPOSE_TYPES
    }


def _merge_values(old, new):

    if any(v is None for v in (old, new)):
        return old or new
    if old == new:
        return old

    old = old if isinstance(old, list) else [old]
    new = new if isinstance(new, list) else [new]
    return list(dict.fromkeys(old + new))


def sanitize_effects_catalog(raw: dict) -> list[dict]:

    merged: dict[int, dict] = {}

    for partition in raw.values():
        for entry in partition:
            target = merged.setdefault(entry["id"], {})
            for key, value in entry.items():
                if value is not None:
                    target[key] = _merge_values(target.get(key), value)

    return [
        {k: v for k, v in entry.items() if v not in (None, "", [])}
        for _, entry in sorted(merged.items())
    ]


raw = fetch_effects_catalog()
merged = sanitize_effects_catalog(raw)
merged

[{'id': 1001,
  'name': '3D环绕',
  'subTitle': '戴上耳机，沉浸式拥抱四方涌来的音乐',
  'playerFloatLayerDesc': '声音从四方传来，耳朵被音乐包裹',
  'useCount': 269512806,
  'trackCount': 34763,
  'type': 1,
  'soundType': 1,
  'animation': 0,
  'size': 113,
  'md5': 'ab0a5aa90b672000fffe580d0c35ea9f',
  'link': 'http://d1.music.126.net/dmusic/d7140bb26fef874e033b1c851297edc2.ncae',
  'version': '5.4.0',
  'newAdd': 0,
  'soundMaterialList': [{'size': 373,
    'md5': '403af171cb2f8af205925e20741a3913',
    'link': 'http://d1.music.126.net/dmusic/obj/w57Cg8KTwrbClD_CmMK-/77258903316/cd70/7ee1/908c/403af171cb2f8af205925e20741a3913.ncae',
    'type': 1}],
  'soundCategory': 1,
  'exposeDescText': '戴上耳机，沉浸式拥抱四方涌来的音乐',
  'exposeLink': 'https://music.163.com/st/vip/sound-effect-detail/audio-effect?nm_style=sbt&soundId=1001'},
 {'id': 1002,
  'name': '纯净ACG',
  'subTitle': '进入ACG的世界，收获另一番悦耳，适合动漫 OP/ED、游戏原声、ACG 同人音乐等',
  'playerFloatLayerDesc': '适配电子配乐的燃感，人声有活力',
  'useCount': 111798752,
  'trackCount': 14622,
  'type': 1,
  's

In [6]:
EFFECTS_PATH = "/api/usertool/sound/queryAllSound"
EFFECTS_EXPOSE_TYPES = (2, 3, 6, 7)


def fetch_effects_catalog() -> dict[int, list[dict]]:
    return {
        i: eapi_post(EFFECTS_PATH, {"exposeType": i})["data"]
        for i in EFFECTS_EXPOSE_TYPES
    }


def _merge_values(old, new):
    print(f"old: {old}, new: {new}")

    if any(v is None for v in (old, new)):
        return old or new
    if old == new:
        return old

    old = old if isinstance(old, list) else [old]
    new = new if isinstance(new, list) else [new]
    return list(dict.fromkeys(old + new))


def sanitize_effects_catalog(raw: dict) -> list[dict]:

    merged: dict[str, dict] = {}

    for partition in raw.values():
        for entry in partition:
            target = merged.setdefault(entry["name"], {})
            for key, value in entry.items():
                if value is not None:
                    target[key] = _merge_values(target.get(key), value)

    return [
        {k: v for k, v in entry.items() if v not in (None, "", [])}
        for _, entry in sorted(merged.items())
    ]


raw = fetch_effects_catalog()
merged = sanitize_effects_catalog(raw)
merged

old: None, new: 3601400
old: None, new: 华晨宇音效
old: None, new: 专属定制，适合听花花的歌曲
old: None, new: 专属定制，适合听花花的歌曲
old: None, new: 1513
old: None, new: 0
old: None, new: 3
old: None, new: 1
old: None, new: 0
old: None, new: 0
old: None, new: 7e37a01730eb71e01186f9c906621c27
old: None, new: http://d1.music.126.net/dmusic/obj/w57Cg8KTwrbClD_CmMK-/82462197821/3d71/0e57/b8fb/4ffa0d4c4d303569606a6905b6b4ec3d.ncae
old: None, new:  
old: None, new: 
old: None, new: 0
old: None, new: [{'size': 649, 'md5': '4ffa0d4c4d303569606a6905b6b4ec3d', 'link': 'http://d1.music.126.net/dmusic/obj/w57Cg8KTwrbClD_CmMK-/82462197821/3d71/0e57/b8fb/4ffa0d4c4d303569606a6905b6b4ec3d.ncae', 'type': 1}]
old: None, new: 1
old: None, new: https://p5.music.126.net/obj/wovDmcKXw6PCn2_CmsOk/79594987406/325e/60b1/2d34/0b41574ebce0eb9d7a6926f983e7a28f.png
old: None, new: https://p5.music.126.net/obj/wovDmcKXw6PCn2_CmsOk/79594987406/325e/60b1/2d34/0b41574ebce0eb9d7a6926f983e7a28f.png
old: None, new: 适合听花花的歌曲
old: None, new: https:/

TypeError: cannot use 'dict' as a dict key (unhashable type: 'dict')

In [7]:
EFFECTS_PATH = "/api/usertool/sound/queryAllSound"
EFFECTS_EXPOSE_TYPES = (2, 3, 6, 7)


def fetch_effects_catalog() -> dict[int, list[dict]]:
    return {
        i: eapi_post(EFFECTS_PATH, {"exposeType": i})["data"]
        for i in EFFECTS_EXPOSE_TYPES
    }


def _merge_values(old, new):
    print(f"old: {old}, new: {new}")

    if any(v is None for v in (old, new)):
        return old or new
    if old == new:
        return old

    old = old if isinstance(old, list) else [old]
    new = new if isinstance(new, list) else [new]
    return old + [v for v in new if v not in old]


def sanitize_effects_catalog(raw: dict) -> list[dict]:

    merged: dict[str, dict] = {}

    for partition in raw.values():
        for entry in partition:
            target = merged.setdefault(entry["name"], {})
            for key, value in entry.items():
                if value is not None:
                    target[key] = _merge_values(target.get(key), value)

    return [
        {k: v for k, v in entry.items() if v not in (None, "", [])}
        for _, entry in sorted(merged.items())
    ]


raw = fetch_effects_catalog()
merged = sanitize_effects_catalog(raw)
merged

old: None, new: 3601400
old: None, new: 华晨宇音效
old: None, new: 专属定制，适合听花花的歌曲
old: None, new: 专属定制，适合听花花的歌曲
old: None, new: 1513
old: None, new: 0
old: None, new: 3
old: None, new: 1
old: None, new: 0
old: None, new: 0
old: None, new: 7e37a01730eb71e01186f9c906621c27
old: None, new: http://d1.music.126.net/dmusic/obj/w57Cg8KTwrbClD_CmMK-/82462197821/3d71/0e57/b8fb/4ffa0d4c4d303569606a6905b6b4ec3d.ncae
old: None, new:  
old: None, new: 
old: None, new: 0
old: None, new: [{'size': 649, 'md5': '4ffa0d4c4d303569606a6905b6b4ec3d', 'link': 'http://d1.music.126.net/dmusic/obj/w57Cg8KTwrbClD_CmMK-/82462197821/3d71/0e57/b8fb/4ffa0d4c4d303569606a6905b6b4ec3d.ncae', 'type': 1}]
old: None, new: 1
old: None, new: https://p5.music.126.net/obj/wovDmcKXw6PCn2_CmsOk/79594987406/325e/60b1/2d34/0b41574ebce0eb9d7a6926f983e7a28f.png
old: None, new: https://p5.music.126.net/obj/wovDmcKXw6PCn2_CmsOk/79594987406/325e/60b1/2d34/0b41574ebce0eb9d7a6926f983e7a28f.png
old: None, new: 适合听花花的歌曲
old: None, new: https:/

[{'id': [1035400, 3592424],
  'name': '360°环绕',
  'subTitle': '精密算法，360°旋转，音乐行星常伴左右',
  'playerFloatLayerDesc': '实时调整声场分布，环绕感随节奏变化',
  'useCount': [66829600, 0],
  'trackCount': [7231, 0],
  'type': 3,
  'soundType': [1, 3],
  'animation': 0,
  'size': [91, 0],
  'md5': 'aee35b9108e221a8bc57365192c7ea6f',
  'link': ['http://d1.music.126.net/dmusic/73d475f9aed56c56d547dbfa37f232dd.ncae',
   'http://d1.music.126.net/dmusic/obj/wo_Dk8OFw6LCn2vCm8Kx/78242449315/234b/9db2/bb07/aee35b9108e221a8bc57365192c7ea6f.ncae'],
  'version': ['5.4.0', ' '],
  'newAdd': 0,
  'soundMaterialList': [{'size': 91,
    'md5': 'aee35b9108e221a8bc57365192c7ea6f',
    'link': 'http://d1.music.126.net/dmusic/obj/w57Cg8KTwrbClD_CmMK-/62324091054/234b/9db2/bb07/aee35b9108e221a8bc57365192c7ea6f.ncae',
    'type': 1},
   {'size': 91,
    'md5': 'aee35b9108e221a8bc57365192c7ea6f',
    'link': 'http://d1.music.126.net/dmusic/obj/wo_Dk8OFw6LCn2vCm8Kx/78242449315/234b/9db2/bb07/aee35b9108e221a8bc57365192c7ea6f.ncae',
   

In [8]:
EFFECTS_PATH = "/api/usertool/sound/queryAllSound"
EFFECTS_EXPOSE_TYPES = (2, 3, 6, 7)


def fetch_effects_catalog() -> dict[int, list[dict]]:
    return {
        i: eapi_post(EFFECTS_PATH, {"exposeType": i})["data"]
        for i in EFFECTS_EXPOSE_TYPES
    }


def _merge_values(old, new):

    if any(v is None for v in (old, new)):
        return old or new
    if old == new:
        return old

    old = old if isinstance(old, list) else [old]
    new = new if isinstance(new, list) else [new]
    return old + [v for v in new if v not in old]


def sanitize_effects_catalog(raw: dict) -> list[dict]:

    merged: dict[str, dict] = {}

    for partition in raw.values():
        for entry in partition:
            target = merged.setdefault(entry["name"], {})
            for key, value in entry.items():
                if value is not None:
                    target[key] = _merge_values(target.get(key), value)

    # collapse id's into one (pick the smallest)
    for entry in merged.values():
        if "id" in entry and isinstance(entry["id"], list):
            entry["id"] = min(entry["id"])

    return [
        {k: v for k, v in entry.items() if v not in (None, "", [])}
        # for _, entry in sorted(merged.items())
        for entry in sorted(merged.values(), key=lambda e: e.get("id", 0))
    ]


raw = fetch_effects_catalog()
merged = sanitize_effects_catalog(raw)
merged

[{'id': 1001,
  'name': '3D环绕',
  'subTitle': '戴上耳机，沉浸式拥抱四方涌来的音乐',
  'playerFloatLayerDesc': '声音从四方传来，耳朵被音乐包裹',
  'useCount': [269512806, 0],
  'trackCount': [34763, 0],
  'type': 1,
  'soundType': [1, 3],
  'animation': 0,
  'size': [113, 0],
  'md5': ['ab0a5aa90b672000fffe580d0c35ea9f',
   '403af171cb2f8af205925e20741a3913'],
  'link': ['http://d1.music.126.net/dmusic/d7140bb26fef874e033b1c851297edc2.ncae',
   'http://d1.music.126.net/dmusic/obj/wo_Dk8OFw6LCn2vCm8Kx/78242522013/cd70/7ee1/908c/403af171cb2f8af205925e20741a3913.ncae'],
  'version': ['5.4.0', ' '],
  'newAdd': 0,
  'soundMaterialList': [{'size': 373,
    'md5': '403af171cb2f8af205925e20741a3913',
    'link': 'http://d1.music.126.net/dmusic/obj/w57Cg8KTwrbClD_CmMK-/77258903316/cd70/7ee1/908c/403af171cb2f8af205925e20741a3913.ncae',
    'type': 1},
   {'size': 373,
    'md5': '403af171cb2f8af205925e20741a3913',
    'link': 'http://d1.music.126.net/dmusic/obj/wo_Dk8OFw6LCn2vCm8Kx/78242522013/cd70/7ee1/908c/403af171cb2f8af205

In [10]:
EFFECTS_PATH = "/api/usertool/sound/queryAllSound"
EFFECTS_EXPOSE_TYPES = (2, 3, 6, 7)


def fetch_effects_catalog() -> dict[int, list[dict]]:
    return {
        i: eapi_post(EFFECTS_PATH, {"exposeType": i})["data"]
        for i in EFFECTS_EXPOSE_TYPES
    }


def _merge_values(old, new):

    # if any(v is None for v in (old, new)):
    #     return old or new
    # # deal with zeros
    # if isinstance(old, int) and isinstance(new, int) and (old == 0 or new == 0):
    #     return old or new
    if any(v is None or v == 0 for v in (old, new)):
        return old or new
    if old == new:
        return old

    old = old if isinstance(old, list) else [old]
    new = new if isinstance(new, list) else [new]
    return old + [v for v in new if v not in old]


def sanitize_effects_catalog(raw: dict) -> list[dict]:

    merged: dict[str, dict] = {}

    for partition in raw.values():
        for entry in partition:
            target = merged.setdefault(entry["name"], {})
            for key, value in entry.items():
                if value is not None:
                    target[key] = _merge_values(target.get(key), value)

    # collapse id's into one (pick the smallest)
    for entry in merged.values():
        if "id" in entry and isinstance(entry["id"], list):
            entry["id"] = min(entry["id"])

    return [
        {k: v for k, v in entry.items() if v not in (None, "", [])}
        # for _, entry in sorted(merged.items())
        for entry in sorted(merged.values(), key=lambda e: e.get("id", 0))
    ]


raw = fetch_effects_catalog()
merged = sanitize_effects_catalog(raw)
merged

[{'id': 1001,
  'name': '3D环绕',
  'subTitle': '戴上耳机，沉浸式拥抱四方涌来的音乐',
  'playerFloatLayerDesc': '声音从四方传来，耳朵被音乐包裹',
  'useCount': 269512806,
  'trackCount': 34763,
  'type': 1,
  'soundType': [1, 3],
  'animation': 0,
  'size': 113,
  'md5': ['ab0a5aa90b672000fffe580d0c35ea9f',
   '403af171cb2f8af205925e20741a3913'],
  'link': ['http://d1.music.126.net/dmusic/d7140bb26fef874e033b1c851297edc2.ncae',
   'http://d1.music.126.net/dmusic/obj/wo_Dk8OFw6LCn2vCm8Kx/78242522013/cd70/7ee1/908c/403af171cb2f8af205925e20741a3913.ncae'],
  'version': ['5.4.0', ' '],
  'newAdd': 0,
  'soundMaterialList': [{'size': 373,
    'md5': '403af171cb2f8af205925e20741a3913',
    'link': 'http://d1.music.126.net/dmusic/obj/w57Cg8KTwrbClD_CmMK-/77258903316/cd70/7ee1/908c/403af171cb2f8af205925e20741a3913.ncae',
    'type': 1},
   {'size': 373,
    'md5': '403af171cb2f8af205925e20741a3913',
    'link': 'http://d1.music.126.net/dmusic/obj/wo_Dk8OFw6LCn2vCm8Kx/78242522013/cd70/7ee1/908c/403af171cb2f8af205925e20741a3913.